# Example 2 — Benchmark evaluation

This notebook reproduces the project's **headline benchmark
numbers**: the six canonical empirical phase-prediction rules
(four textbook rules plus the v1.1 phi-family rules from King 2016
and Ye 2015) evaluated as diagnostic classifiers against the
consolidated v0.1.0 dataset. It also shows the v1.1
intermetallic-aware sub-benchmark that uses Peivaste's 12-class
side-channel labels to score the phi rules against an explicit
solid-solution-versus-intermetallic ground truth.

Every number below is **pinned in `tests/test_evaluate.py`,
`tests/test_evaluate_phi.py`, and `tests/test_intermetallic_subbench.py`**
as ground truth, so any future code or data change that perturbs them
also breaks the test suite.

## Load the rule baselines

`evaluate.build_report` loads the consolidated benchmark, runs each
rule against the appropriate subset of alloys, and returns a dict
with all the diagnostic statistics.

In [1]:
from hea_bench.evaluate import build_report, format_report

report = build_report(include_phi=True)
print(f"benchmark CSV:        {report['csv_path']}")
print(f"non-conflict rows:    {report['n_rows_loaded']}")
print(f"rules evaluated:      {list(report['rules'])}")

benchmark CSV:        D:\Documents\Visual studio\HTMLs\hea-bench\data\consolidated\v0.1.0\consolidated.csv
non-conflict rows:    7684
rules evaluated:      ['zhang_delta_6_5', 'yang_omega_1_1', 'guo_vec_stratified', 'yeh_smix_descriptive', 'king_phi_1_0', 'ye_phi_20_0']

## Headline numbers — formatted summary

In [2]:
print(format_report(report))

=== Rule baselines on D:\Documents\Visual studio\HTMLs\hea-bench\data\consolidated\v0.1.0\consolidated.csv ===
compositions with canonical_phase (non-conflict): 7684

--- Zhang δ < 6.5% (binary single vs multi) ---
  n_evaluated = 6922   accuracy = 57.1%  [95% CI 55.9%, 58.3%]
  sensitivity (single-phase) = 98.9%
  specificity (multi-phase)  = 10.5%
  Youden's J = 0.094

--- Yang Ω > 1.1 (binary single vs multi) ---
  n_evaluated = 6922   accuracy = 54.2%  [95% CI 53.0%, 55.3%]
  sensitivity (single-phase) = 95.0%
  specificity (multi-phase)  = 8.6%
  Youden's J = 0.036

--- King Φ > 1.0 (binary solid_solution vs intermetallic; default T=T_m) ---
  n_evaluated = 6922   accuracy = 48.9%  [95% CI 47.7%, 50.1%]
  sensitivity (solid_solution) = 82.7%
  specificity (intermetallic)  = 11.2%
  Youden's J = -0.061

--- Ye φ > 20.0 (binary solid_solution vs intermetallic) ---
  n_evaluated = 6922   accuracy = 49.1%  [95% CI 47.9%, 50.3%]
  sensitivity (solid_solution) = 44.2%
  specificity (int

## What the numbers mean

Both binary rules — Zhang δ < 6.5% and Yang Ω > 1.1 — have very
high sensitivity (they almost never miss a true single-phase
alloy) but very low specificity (they almost never correctly
identify a multi-phase alloy). Both have Youden's J close to zero,
meaning that as classifiers they are barely better than
always-predict-single-phase. **This is the publishable
observation:** the canonical rules generalize poorly across the
broader open HEA literature.

The Guo–Liu VEC rule, evaluated stratified to single-phase
observations (BCC or FCC only — it was never designed to predict
*whether* a composition forms single-phase, only *which* crystal
structure if it does), achieves 66.9% accuracy with a strong FCC
bias: it catches 92.3% of FCC alloys but only 48.3% of BCC alloys.

## Looking at individual rule details

In [3]:
zhang = report["rules"]["zhang_delta_6_5"]
print(f"Zhang δ < 6.5% rule details:")
print(f"  n evaluated      : {zhang['n']}")
print(f"  true positives   : {zhang['true_positive']}  (predicted single, observed single)")
print(f"  false positives  : {zhang['false_positive']}  (predicted single, observed multi)")
print(f"  true negatives   : {zhang['true_negative']}  (predicted multi,  observed multi)")
print(f"  false negatives  : {zhang['false_negative']}  (predicted multi,  observed single)")
print(f"  ----")
print(f"  accuracy   = {zhang['accuracy']:.3f}  [95% CI {zhang['accuracy_ci95'][0]:.3f}, {zhang['accuracy_ci95'][1]:.3f}]")
print(f"  sensitivity = {zhang['sensitivity']:.3f}")
print(f"  specificity = {zhang['specificity']:.3f}")

Zhang δ < 6.5% rule details:
  n evaluated      : 6922
  true positives   : 3609  (predicted single, observed single)
  false positives  : 2928  (predicted single, observed multi)
  true negatives   : 344  (predicted multi,  observed multi)
  false negatives  : 41  (predicted multi,  observed single)
  ----
  accuracy   = 0.571  [95% CI 0.559, 0.583]
  sensitivity = 0.989
  specificity = 0.105

The 6,030 false positives are the story. Most multi-phase alloys
in the benchmark have δ values *below* 6.5%, so the canonical
threshold predicts them as single-phase. The rule is well-suited
to confirm a single-phase candidate but poorly suited to *rule
out* multi-phase formation.

## A ROC sweep — what threshold would actually work?

`roc_sweep` lets us try every threshold from 1.0% to 12.0% (in
0.1% steps) and see what the best classifier-style trade-off would
be on this dataset.

In [4]:
import csv
import pathlib

from hea_bench.classifiers.diagnostic_stats import roc_sweep
from hea_bench.composition import parse_formula
from hea_bench.descriptors.size import delta
from hea_bench.descriptors.data.elemental import covered_elements
from hea_bench.evaluate import _binary_observed

V010 = pathlib.Path(report["csv_path"])
elemental = covered_elements()

delta_values, observations = [], []
with V010.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        canonical = row.get("canonical_phase", "").strip()
        if not canonical:
            continue
        comp = parse_formula(row["composition_key"])
        if not set(comp).issubset(elemental):
            continue
        delta_values.append(delta(comp))
        observations.append(_binary_observed(canonical))

thresholds = [t / 10.0 for t in range(10, 121)]
points = roc_sweep(delta_values, observations, thresholds, "single-phase", positive_below_threshold=True)
best_acc = max(points, key=lambda p: p.accuracy)
best_j   = max(points, key=lambda p: p.youden_j)

print(f"ROC sweep over Zhang δ rule, {len(points)} threshold points evaluated.")
print(f"  best accuracy    : {best_acc.accuracy:.3f} at threshold δ < {best_acc.threshold}%")
print(f"    (sens = {best_acc.sensitivity:.3f}, spec = {best_acc.specificity:.3f})")
print(f"  best Youden's J  : J = {best_j.youden_j:.3f} at threshold δ < {best_j.threshold}%")
print(f"    (sens = {best_j.sensitivity:.3f}, spec = {best_j.specificity:.3f})")
print(f"  canonical 6.5% threshold for reference: J = {next(p.youden_j for p in points if p.threshold == 6.5):.3f}")

ROC sweep over Zhang δ rule, 111 threshold points evaluated.
  best accuracy    : 0.572 at threshold δ < 6.6%
    (sens = 0.993, spec = 0.102)
  best Youden's J  : J = 0.123 at threshold δ < 2.6%
    (sens = 0.260, spec = 0.863)
  canonical 6.5% threshold for reference: J = 0.094

The optimal threshold on this dataset is *much tighter* than the
canonical 6.5%. The accuracy at the optimum is materially higher
than at the canonical value, but it's still nowhere near a
strong classifier — the empirical rule has real but limited
power to separate single-phase from multi-phase compositions
across the full open literature.

This kind of recalibration finding is exactly the use that
`hea-bench` is designed to support.

## v1.1 intermetallic-aware sub-benchmark

The main benchmark collapses intermetallic alloys into a single
"multi-phase" class. The Peivaste source ships a finer 12-class
label that separates intermetallic-containing alloys (IM, FCC+IM,
BCC+IM, BCC+FCC+IM) from pure solid-solution states (BCC, FCC,
HCP, BCC+FCC) and from amorphous states. v1.1 uses those labels
to construct a sub-benchmark that scores the phi-family rules
against an explicit `solid_solution` versus `intermetallic`
ground truth.

In [5]:
from hea_bench.evaluate import build_intermetallic_subbench_report

sub = build_intermetallic_subbench_report()
print(f"sub-benchmark rows loaded:  {sub['n_rows_loaded']}")
print()
print("In-sample at the published cutoff:")
for rule, stats in sub["in_sample"].items():
    print(
        f"  {rule:18s} n={stats['n']}  "
        f"acc={stats['accuracy']:.3f}  "
        f"sens={stats['sensitivity']:.3f}  "
        f"spec={stats['specificity']:.3f}  "
        f"J={stats['youden_j']:+.3f}"
    )
print()
print("Held-out 5-fold with per-fold tuned cutoffs:")
for rule, summary in sub["holdout_tuned"].items():
    print(
        f"  {rule:18s} acc={summary['accuracy_mean']:.3f}  "
        f"J={summary['youden_j_mean']:+.3f}  "
        f"threshold mean={summary['threshold_mean']:.2f}"
    )

sub-benchmark rows loaded:  5930

In-sample at the published cutoff:
  king_phi_1_0       n=5685  acc=0.731  sens=0.842  spec=0.143  J=-0.015
  ye_phi_20_0        n=5685  acc=0.491  sens=0.443  spec=0.748  J=+0.191

Held-out 5-fold with per-fold tuned cutoffs:
  king_phi_tuned     acc=0.263  J=+0.019  threshold mean=3.62
  ye_phi_tuned       acc=0.626  J=+0.206  threshold mean=12.00

The sub-benchmark is what makes Ye φ's signal visible. On the
coarse main benchmark Ye φ has Youden's J ≈ -0.03 (worse than
random) at its published cutoff. On the intermetallic-aware
sub-benchmark it has J ≈ +0.17, an order of magnitude larger and
clearly positive. The Peivaste 12-class label is restoring a
distinction that the lowest-common-denominator main taxonomy
hides. King Φ remains weak on both, consistent with v1.1's
raw-pair-enthalpy approximation of ΔG_max rather than the full
ordered-compound Miedema branch used by King 2016.